# 15 — B2 RadImageNet sampled-neighbor 2.5D, five-fold group CV

Controlled input-representation experiment against notebook 09 B2. Keep the
same frozen RadImageNet ResNet50 checkpoint, 05A 224×224 cache, five
report-group folds, L1 soft labels, exact MIL head, optimizer and weak-BCE
checkpoint selection. The **only scientific change** is replacing repeated
grayscale RGB with `[previous sampled slice, current slice, next sampled
slice]` within each of the five MRI slots. Edge slices repeat themselves.
These are neighbors in the existing 05A **sampled cache**, not necessarily
immediately adjacent original DICOM slices. Gold58 remains audit-only.

Attach: RSNA competition; notebook 02 Output containing
`L1_family_mean_RECOMMENDED.csv`; private `Study_case 05_RSNA` Dataset with
complete `rsna_05a_dense_mil/study_cache` and `fold_assignments.csv`; public
`resnet-50-radimagenet-marwan` Dataset with `ResNet50.pt`. Internet Off.
First run the asset cell and CPU neighbor-mapping smoke-test cell. They must
print `ASSET CHECK PASSED` and `NEIGHBOR SMOKE PASSED`. Then switch to T4×2
and Save & Run All. This notebook uses GPU 0 only, makes **no submission**,
and has an internal 3-hour safety guard, not a promised runtime.

Send the Output ZIP containing `rsna_b2_sampled25d5/run_report.json`,
`weak_oof.csv`, `cv_metrics.csv`, `gold_metrics.csv`, and five head checkpoints.
Do not blend it into the 0.943 submission until its paired OOF comparison
against notebook 09 has been audited.

In [ ]:
from pathlib import Path
from time import monotonic
import copy, gc, hashlib, json, os, random
import numpy as np
import pandas as pd
import psutil
from sklearn.metrics import roc_auc_score
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision.models import resnet50

LABELS = ['ACL','MCL','Medial Meniscus','Lateral Meniscus','Medial OA','Lateral OA',
          'PF OA','Effusion','Synovitis',"Baker's",'Contusion','Fracture']
CONFIG = {
    'experiment': 'B2_RadImageNet_sampled_neighbor_25D_fivefold_v1', 'seed': 111,
    'active_folds': [0,1,2,3,4], 'image_microbatch': 24, 'max_tokens': 80,
    'feature_dim': 2048, 'head_dim': 256, 'head_batch': 32,
    'channel_mode': 'previous_sampled_current_next_sampled_within_slot',
    'max_epochs': 20, 'patience': 4, 'head_lr': 3e-4, 'weight_decay': 1e-4,
    'min_available_ram_gib': 5.0, 'max_total_hours': 3.0,
    'encoder_sha256': '08629f7e7bd3e29b8ee9522ca3f65ce4d010a7ddf74f0ea3c7e3f3d0bbab0734',
}
INPUT = Path('/kaggle/input')
OUT = Path('/kaggle/working/rsna_b2_sampled25d5')
OUT.mkdir(parents=True, exist_ok=True)
START = monotonic()
random.seed(CONFIG['seed']); np.random.seed(CONFIG['seed']); torch.manual_seed(CONFIG['seed'])
torch.backends.cudnn.benchmark = False

def shallow_files(name, max_depth=8):
    found = []
    for base, dirs, files in os.walk(INPUT):
        depth = len(Path(base).relative_to(INPUT).parts)
        dirs[:] = [d for d in dirs if d not in {'train_series','test_series','study_cache','dicom','dicoms'}]
        if name in files: found.append(Path(base) / name)
        if depth >= max_depth: dirs[:] = []
    return sorted(found, key=str)

def cache_directories(max_depth=9):
    found = []
    for base, dirs, _ in os.walk(INPUT):
        depth = len(Path(base).relative_to(INPUT).parts)
        dirs[:] = [d for d in dirs if d not in {'train_series','test_series','dicom','dicoms'}]
        if 'study_cache' in dirs:
            found.append(Path(base) / 'study_cache'); dirs.remove('study_cache')
        if depth >= max_depth: dirs[:] = []
    return found

cache_counts = []
for d in cache_directories():
    names = [e.name for e in os.scandir(d) if e.is_file()]
    cache_counts.append((d, sum(n.startswith('weak_') and n.endswith('.npz') for n in names),
                         sum(n.startswith('gold_') and n.endswith('.npz') for n in names)))
print('CACHE:', [(str(p),w,g) for p,w,g in cache_counts])
valid = [(p,w,g) for p,w,g in cache_counts if w >= 4349 and g >= 58]
if not valid: raise FileNotFoundError('Attach Study_case 05_RSNA Dataset with complete 05A study_cache; no encoding started')
CACHE = sorted(valid, key=lambda x: (-x[1],str(x[0])))[0][0]
WEIGHT_CANDIDATES = [INPUT/'datasets/marwanmath/resnet-50-radimagenet-marwan/ResNet50.pt',
                     INPUT/'resnet-50-radimagenet-marwan/ResNet50.pt']
WEIGHT = next((p for p in WEIGHT_CANDIDATES if p.is_file()), None)
if WEIGHT is None: raise FileNotFoundError('Attach resnet-50-radimagenet-marwan Dataset')
digest = hashlib.sha256()
with WEIGHT.open('rb') as fh:
    for block in iter(lambda: fh.read(8<<20), b''): digest.update(block)
if digest.hexdigest() != CONFIG['encoder_sha256']:
    raise RuntimeError('RadImageNet weight SHA256 mismatch')
COMP = next((p for p in [INPUT/'competitions/rsna-knee-abnormality-detection',
                       INPUT/'rsna-knee-abnormality-detection'] if (p/'train.csv').is_file()), None)
if COMP is None: raise FileNotFoundError('Attach RSNA competition input')
TRAIN = pd.read_csv(COMP/'train.csv', dtype={'StudyInstanceUID':str})
if len(TRAIN) != 4407 or not TRAIN.StudyInstanceUID.is_unique: raise RuntimeError('Unexpected train.csv identity')
gold = TRAIN.loc[TRAIN[LABELS].notna().all(axis=1), ['StudyInstanceUID']+LABELS].copy()
if len(gold) != 58: raise RuntimeError('Expected exactly 58 structured gold studies')
SILVER_FILES = [p for p in shallow_files('L1_family_mean_RECOMMENDED.csv')
                if set(['StudyInstanceUID']+LABELS).issubset(pd.read_csv(p,nrows=1).columns)]
if not SILVER_FILES: raise FileNotFoundError('Attach notebook 02 Output containing L1_family_mean_RECOMMENDED.csv')
if len(SILVER_FILES) > 1: print('Multiple silver copies; using:', SILVER_FILES[0])
SILVER_PATH = SILVER_FILES[0]
silver = pd.read_csv(SILVER_PATH, dtype={'StudyInstanceUID':str})
if len(silver) != 4349 or not silver.StudyInstanceUID.is_unique: raise RuntimeError('Silver IDs/count mismatch')
if set(silver.StudyInstanceUID) & set(gold.StudyInstanceUID): raise RuntimeError('Gold leaked into weak silver')
if set(silver.StudyInstanceUID) | set(gold.StudyInstanceUID) != set(TRAIN.StudyInstanceUID):
    raise RuntimeError('Silver+gold does not cover all train studies')
if not np.isfinite(silver[LABELS].to_numpy(float)).all(): raise RuntimeError('Nonfinite silver labels')
if ((silver[LABELS] < 0) | (silver[LABELS] > 1)).any().any(): raise RuntimeError('Silver probabilities outside [0,1]')
FOLD_FILES = [p for p in shallow_files('fold_assignments.csv')
              if set(['StudyInstanceUID','fold','report_group']).issubset(pd.read_csv(p,nrows=1).columns)]
FOLD_FILES = sorted(FOLD_FILES, key=lambda p: (str(CACHE.parent.parent) not in str(p), str(p)))
if not FOLD_FILES: raise FileNotFoundError('Attach 05A output containing grouped fold_assignments.csv')
FOLD_PATH = FOLD_FILES[0]
folds = pd.read_csv(FOLD_PATH, dtype={'StudyInstanceUID':str})[['StudyInstanceUID','report_group','fold']]
if not folds.StudyInstanceUID.is_unique: raise RuntimeError('Duplicate fold IDs')
weak = silver[['StudyInstanceUID']+LABELS].merge(folds,on='StudyInstanceUID',how='left',validate='one_to_one')
if weak.fold.isna().any() or weak.report_group.isna().any(): raise RuntimeError('Missing weak fold/group')
weak['fold'] = weak.fold.astype(int)
if set(weak.fold) != set(range(5)): raise RuntimeError('Expected unchanged 05A folds 0–4')
for f in range(5):
    tr = weak.loc[weak.fold.ne(f),'report_group'].astype(str)
    va = weak.loc[weak.fold.eq(f),'report_group'].astype(str)
    if not set(tr).isdisjoint(set(va)): raise RuntimeError(f'Report-group leakage in fold {f}')
IDS = weak.StudyInstanceUID.astype(str).tolist() + gold.StudyInstanceUID.astype(str).tolist()
Y = np.vstack([weak[LABELS].to_numpy(np.float32), gold[LABELS].to_numpy(np.float32)])
if Y.shape != (4407,12): raise RuntimeError('Target matrix shape drift')
missing_cache = [uid for i,uid in enumerate(IDS)
                 if not (CACHE / f'{"weak" if i < len(weak) else "gold"}_{uid}.npz').is_file()]
if missing_cache: raise FileNotFoundError(f'{len(missing_cache)} cache studies missing, first: {missing_cache[:3]}')
print('ASSET CHECK PASSED', {'weak':len(weak),'gold':len(gold),'folds':weak.fold.value_counts().sort_index().to_dict(),
                             'silver':str(SILVER_PATH),'fold_file':str(FOLD_PATH),'cache':str(CACHE)})


In [ ]:
def neighbors_within_slot(slot_ids):
    slot_ids = np.asarray(slot_ids)
    previous = np.arange(len(slot_ids),dtype=np.int32)
    following = previous.copy()
    for slot in range(5):
        idx = np.flatnonzero(slot_ids == slot)
        if idx.size:
            previous[idx] = np.r_[idx[0],idx[:-1]]
            following[idx] = np.r_[idx[1:],idx[-1]]
    return previous,following

# CPU-only test: channel neighbors must never cross an MRI slot boundary.
test_slots = np.array([0,0,2,2,4,4,1],dtype=np.int64)
test_images = np.arange(len(test_slots),dtype=np.uint8)[:,None,None]
test_prev,test_next = neighbors_within_slot(test_slots)
test_rgb = np.stack([test_images[test_prev],test_images,test_images[test_next]],axis=1)
assert test_prev.tolist() == [0,0,2,2,4,4,6]
assert test_next.tolist() == [1,1,3,3,5,5,6]
assert test_rgb[:,1,0,0].tolist() == list(range(7))
assert np.all(test_slots[test_prev] == test_slots)
assert np.all(test_slots[test_next] == test_slots)
del test_slots,test_images,test_prev,test_next,test_rgb
print('NEIGHBOR SMOKE PASSED: center retained, edges repeated, no cross-slot mixing',flush=True)

In [ ]:
if not torch.cuda.is_available() or 'T4' not in torch.cuda.get_device_name(0):
    raise RuntimeError('Asset check passed; select T4x2 before feature extraction')
DEVICE = torch.device('cuda:0')

class RadEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = nn.Sequential(*list(resnet50(weights=None).children())[:-2])
    def forward(self,x): return self.backbone(x).mean(dim=(2,3))

feature_bytes = len(IDS)*CONFIG['max_tokens']*CONFIG['feature_dim']*2
available = psutil.virtual_memory().available
if available < feature_bytes + 8*2**30:
    raise RuntimeError(f'Not enough available host RAM before feature allocation: {available/2**30:.2f} GiB')
features = np.zeros((len(IDS),CONFIG['max_tokens'],CONFIG['feature_dim']),np.float16)
slot_ids = np.zeros((len(IDS),CONFIG['max_tokens']),np.int64)
positions = np.zeros((len(IDS),CONFIG['max_tokens']),np.float32)
masks = np.zeros((len(IDS),CONFIG['max_tokens']),bool)
encoder = RadEncoder()
state = torch.load(WEIGHT,map_location='cpu',weights_only=True)
expected = encoder.state_dict()
missing = set(expected)-set(state); extra = set(state)-set(expected)
bad = [k for k in set(expected)&set(state) if expected[k].shape != state[k].shape]
if missing or extra or bad: raise RuntimeError(f'Encoder checkpoint mismatch: {len(missing)} missing, {len(extra)} extra, {len(bad)} shapes')
encoder.load_state_dict(state,strict=True)
del state,expected; gc.collect()
encoder = encoder.to(DEVICE).eval().requires_grad_(False)
torch.cuda.reset_peak_memory_stats(0)
encode_start = monotonic(); n_images = 0; empty = 0
with torch.inference_mode():
    for i,uid in enumerate(IDS):
        split = 'weak' if i < len(weak) else 'gold'
        path = CACHE/f'{split}_{uid}.npz'
        if not path.is_file(): raise FileNotFoundError(f'Missing required 05A cache: {path}')
        with np.load(path,allow_pickle=False) as z:
            if not {'images','slot_ids','positions','slice_indices'}.issubset(z.files):
                raise RuntimeError(f'Invalid cache keys: {path}')
            images=z['images']; s=z['slot_ids']; pos=z['positions']
            if images.dtype!=np.uint8 or images.ndim!=3 or images.shape[1:]!=(224,224):
                raise RuntimeError(f'Invalid cache image shape/dtype: {path}')
            t=len(images)
            if t>CONFIG['max_tokens'] or len(s)!=t or len(pos)!=t or len(z['slice_indices'])!=t:
                raise RuntimeError(f'Invalid cache token count: {path}')
            if t and (s.min()<0 or s.max()>=5 or not np.isfinite(pos).all()):
                raise RuntimeError(f'Invalid cache slot/position metadata: {path}')
        if t==0: empty+=1
        prev_idx,next_idx=neighbors_within_slot(s)
        slot_ids[i,:t]=s; positions[i,:t]=pos; masks[i,:t]=True; n_images+=t
        for offset in range(0,t,CONFIG['image_microbatch']):
            part=images[offset:offset+CONFIG['image_microbatch']]
            selected=np.arange(offset,offset+len(part))
            triplet=np.stack([images[prev_idx[selected]],part,
                              images[next_idx[selected]]],axis=1)
            rgb=torch.from_numpy(triplet).to(DEVICE).float().div_(127.5).sub_(1.0)
            with torch.autocast('cuda',dtype=torch.float16): value=encoder(rgb)
            if value.shape!=(len(part),2048) or not bool(torch.isfinite(value).all()):
                raise RuntimeError(f'Nonfinite/wrong encoder output: {path}')
            half=value.to(torch.float16).cpu().numpy()
            if not np.isfinite(half).all(): raise RuntimeError(f'Feature half overflow: {path}')
            features[i,offset:offset+len(part)]=half
            del selected,triplet,rgb,value,half
        del images,s,pos,prev_idx,next_idx
        if i==63:
            torch.cuda.synchronize(0)
            early_rate=n_images/max(1e-6,monotonic()-encode_start)
            print('EARLY 2.5D RATE',round(early_rate,1),'images/sec',flush=True)
            if early_rate < 75:
                raise RuntimeError('2.5D encoder too slow; stopped after 64 studies')
        if (i+1)%250==0:
            torch.cuda.synchronize(0)
            print(f'encoded {i+1}/{len(IDS)} studies; {n_images} images; {monotonic()-encode_start:.1f}s',flush=True)
            if psutil.virtual_memory().available/2**30 < CONFIG['min_available_ram_gib']:
                raise RuntimeError('Host RAM safety threshold reached during feature extraction')
            if (monotonic()-START)/3600 > CONFIG['max_total_hours']:
                raise RuntimeError('Overall configured time safety limit reached during feature extraction')
torch.cuda.synchronize(0)
encode_seconds=monotonic()-encode_start
encoder_peak_gib=torch.cuda.max_memory_allocated(0)/2**30
del encoder; gc.collect(); torch.cuda.empty_cache()
if empty: raise RuntimeError(f'{empty} empty studies; refusing training')
if not np.isfinite(features[masks]).all(): raise RuntimeError('Nonfinite feature array')
print('FEATURE EXTRACTION COMPLETE', {'seconds':round(encode_seconds,1),'images':n_images,
                                     'empty_studies':empty,'host_feature_gib':round(feature_bytes/2**30,3),
                                     'encoder_peak_gpu_gib':round(encoder_peak_gib,3)})


In [ ]:
class FeatureDataset(Dataset):
    def __init__(self,indices): self.indices=np.asarray(indices,dtype=int)
    def __len__(self): return len(self.indices)
    def __getitem__(self,k):
        i=int(self.indices[k])
        return (i,torch.from_numpy(features[i]),torch.from_numpy(slot_ids[i]),
                torch.from_numpy(positions[i]),torch.from_numpy(masks[i]),torch.from_numpy(Y[i]))

class StudyMIL(nn.Module):
    def __init__(self):
        super().__init__(); d=CONFIG['head_dim']
        self.project=nn.Sequential(nn.LayerNorm(2048),nn.Linear(2048,d),nn.GELU(),nn.Dropout(.10))
        self.slot=nn.Embedding(5,d)
        self.position=nn.Sequential(nn.Linear(1,d),nn.GELU(),nn.Linear(d,d))
        self.query=nn.Parameter(torch.randn(12,d)*.02)
        self.attention=nn.MultiheadAttention(d,4,dropout=.10,batch_first=True)
        self.norm=nn.LayerNorm(d)
        self.weight=nn.Parameter(torch.randn(12,d)*.02); self.bias=nn.Parameter(torch.zeros(12))
    def forward(self,x,s,p,m):
        m=m.clone(); m[~m.any(1),0]=True
        token=self.project(x.float())+self.slot(s)+self.position(p.unsqueeze(-1))
        q=self.query.unsqueeze(0).expand(len(x),-1,-1)
        pooled,_=self.attention(q,token,token,key_padding_mask=~m,need_weights=False)
        return (self.norm(pooled)*self.weight.unsqueeze(0)).sum(-1)+self.bias

def loader(indices,shuffle=False,seed=0):
    g=torch.Generator().manual_seed(CONFIG['seed']+seed)
    return DataLoader(FeatureDataset(indices),batch_size=CONFIG['head_batch'],shuffle=shuffle,
                      generator=g,num_workers=0,pin_memory=False,drop_last=False)

def pass_batches(model,data,optimizer=None):
    training=optimizer is not None
    model.train(training)
    ids=[]; probs=[]; loss_sum=0.0; count=0
    context=torch.enable_grad() if training else torch.inference_mode()
    with context:
        for ix,x,s,p,m,y in data:
            x=x.to(DEVICE).float(); s=s.to(DEVICE); p=p.to(DEVICE); m=m.to(DEVICE); y=y.to(DEVICE)
            logits=model(x,s,p,m)
            loss=F.binary_cross_entropy_with_logits(logits,y)
            if training:
                optimizer.zero_grad(set_to_none=True); loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(),1.0); optimizer.step()
            loss_sum+=float(loss.detach())*len(ix); count+=len(ix)
            if not training:
                ids.extend(ix.tolist()); probs.append(torch.sigmoid(logits).cpu().numpy())
    return loss_sum/max(1,count),np.asarray(ids,dtype=int),np.concatenate(probs) if probs else None

def target_auc(y,p):
    out=[]
    for j in range(12):
        yy=y[:,j]
        out.append(float(roc_auc_score(yy,p[:,j])) if len(np.unique(yy))==2 else np.nan)
    return out

history=[]; cv_rows=[]; gold_rows=[]; oof_parts=[]; gold_parts=[]
gold_indices=np.arange(len(weak),len(IDS))
gold_loader=loader(gold_indices)
for fold in CONFIG['active_folds']:
    if (monotonic()-START)/3600 > CONFIG['max_total_hours']:
        print('TIME GUARD: stopping before next fold'); break
    torch.manual_seed(CONFIG['seed']+fold)
    tr=np.flatnonzero(weak.fold.to_numpy()!=fold)
    va=np.flatnonzero(weak.fold.to_numpy()==fold)
    if not set(weak.iloc[tr].report_group.astype(str)).isdisjoint(set(weak.iloc[va].report_group.astype(str))):
        raise RuntimeError(f'Fold {fold} report-group leakage')
    tr_loader=loader(tr,shuffle=True,seed=fold); va_loader=loader(va)
    model=StudyMIL().to(DEVICE)
    optimizer=torch.optim.AdamW(model.parameters(),lr=CONFIG['head_lr'],weight_decay=CONFIG['weight_decay'])
    best_bce=float('inf'); best_epoch=0; best_state=None; stale=0; fold_stop_reason=None
    for epoch in range(1,CONFIG['max_epochs']+1):
        if (monotonic()-START)/3600 > CONFIG['max_total_hours']:
            fold_stop_reason='time_guard'; print('TIME GUARD: ending fold with best checkpoint so far'); break
        t0=monotonic()
        train_bce,_,_=pass_batches(model,tr_loader,optimizer)
        val_bce,vi,vp=pass_batches(model,va_loader)
        auc=target_auc((Y[vi]>=.5).astype(int),vp)
        row={'fold':fold,'epoch':epoch,'train_bce':train_bce,'val_bce':val_bce,
             'diagnostic_weak_macro_auc':float(np.nanmean(auc)),'seconds':monotonic()-t0}
        history.append(row); pd.DataFrame(history).to_csv(OUT/'training_history.csv',index=False)
        print(row,flush=True)
        if val_bce < best_bce-1e-5:
            best_bce=val_bce; best_epoch=epoch; stale=0
            best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
        else: stale+=1
        if psutil.virtual_memory().available/2**30 < CONFIG['min_available_ram_gib']:
            fold_stop_reason='host_ram_guard'; print('HOST RAM GUARD: ending fold with best checkpoint so far'); break
        if stale>=CONFIG['patience']: break
    if best_state is None: raise RuntimeError(f'Fold {fold} produced no checkpoint')
    model.load_state_dict(best_state)
    torch.save({'state_dict':best_state,'fold':fold,'best_epoch':best_epoch,'best_val_bce':best_bce,
                'config':CONFIG,'labels':LABELS,'source_weight_sha256':digest.hexdigest()},OUT/f'fold_{fold}_head.pt')
    val_bce,vi,vp=pass_batches(model,va_loader)
    if len(vi)!=len(va) or set(vi)!=set(va): raise RuntimeError(f'Fold {fold} validation identity mismatch')
    wauc=target_auc((Y[vi]>=.5).astype(int),vp)
    cv_rows.append({'fold':fold,'best_epoch':best_epoch,'val_bce':val_bce,'stop_reason':fold_stop_reason,
                    'weak_macro_auc_proxy':float(np.nanmean(wauc)),'n_val':len(vi)})
    part=pd.DataFrame(vp,columns=LABELS); part.insert(0,'StudyInstanceUID',[IDS[i] for i in vi]); part['fold']=fold
    oof_parts.append(part); pd.concat(oof_parts).to_csv(OUT/'weak_oof.csv',index=False)
    _,gi,gp=pass_batches(model,gold_loader)
    if list(gi)!=list(gold_indices): raise RuntimeError('Gold audit identity mismatch')
    gauc=target_auc(Y[gi].astype(int),gp)
    gold_rows.append({'fold':fold,'gold_macro_auc_audit_only':float(np.nanmean(gauc)),
                      **{f'auc_{LABELS[j]}':gauc[j] for j in range(12)}})
    gold_parts.append(gp)
    pd.DataFrame(cv_rows).to_csv(OUT/'cv_metrics.csv',index=False)
    pd.DataFrame(gold_rows).to_csv(OUT/'gold_metrics.csv',index=False)
    del model,optimizer,best_state,tr_loader,va_loader; gc.collect(); torch.cuda.empty_cache()
if len(cv_rows) == 5 and all(r['stop_reason'] is None for r in cv_rows):
    joined = pd.concat(oof_parts,ignore_index=True)
    if (len(joined) != len(weak) or not joined.StudyInstanceUID.is_unique or
            set(joined.StudyInstanceUID) != set(weak.StudyInstanceUID) or
            len(gold_parts) != 5 or
            not all((OUT/f'fold_{fold}_head.pt').is_file() for fold in range(5))):
        raise RuntimeError('Five-fold output integrity failed; refusing COMPLETE')
if gold_parts:
    mean_gold=np.mean(np.stack(gold_parts),axis=0)
    result=pd.DataFrame(mean_gold,columns=LABELS)
    result.insert(0,'StudyInstanceUID',gold.StudyInstanceUID.tolist())
    result.to_csv(OUT/'gold_predictions.csv',index=False)
    ensemble_auc=target_auc(gold[LABELS].to_numpy(int),mean_gold)
else: ensemble_auc=[np.nan]*12
completed_cleanly=(len(cv_rows)==len(CONFIG['active_folds']) and all(r['stop_reason'] is None for r in cv_rows))
report={'status':'COMPLETE' if completed_cleanly else 'PARTIAL_GUARD',
        'config':CONFIG,'n_weak':len(weak),'n_gold':len(gold),'n_features':len(IDS),
        'n_images_encoded':n_images,'empty_studies':empty,'encode_seconds':encode_seconds,
        'encoder_peak_gpu_gib':encoder_peak_gib,'host_feature_gib':feature_bytes/2**30,
        'total_elapsed_minutes':(monotonic()-START)/60,'folds_completed':[r['fold'] for r in cv_rows],
        'cv_metrics':cv_rows,'gold_metrics':gold_rows,
        'gold_mean_probability_macro_auc_audit_only':float(np.nanmean(ensemble_auc)),
        'gold_mean_probability_per_target_auc_audit_only':dict(zip(LABELS,ensemble_auc)),
        'silver_path':str(SILVER_PATH),'fold_path':str(FOLD_PATH),'cache_path':str(CACHE),
        'encoder_weight_path':str(WEIGHT),'encoder_weight_sha256':digest.hexdigest(),
        'warning':'Gold58 is audit-only, small, and not independent of public ensemble checkpoint selection; no blend selection from this number'}
def json_safe(value):
    if isinstance(value,dict): return {str(k):json_safe(v) for k,v in value.items()}
    if isinstance(value,(list,tuple)): return [json_safe(v) for v in value]
    if isinstance(value,(float,np.floating)) and not np.isfinite(value): return None
    if isinstance(value,np.integer): return int(value)
    return value
(OUT/'run_report.json').write_text(json.dumps(json_safe(report),indent=2,allow_nan=False),encoding='utf-8')
print('RUN REPORT:',json.dumps({'status':report['status'],'encode_minutes':round(encode_seconds/60,1),
      'folds_completed':report['folds_completed'],'cv_metrics':cv_rows,
      'gold_macro_auc_audit_only':report['gold_mean_probability_macro_auc_audit_only'],
      'elapsed_minutes':round(report['total_elapsed_minutes'],1)},indent=2))
